In [0]:
dbutils.widgets.text("p_batch_id", "")
v_batch_id = dbutils.widgets.get("p_batch_id")

In [0]:
%run "../00-common/01.environment-config"

In [0]:
%run "../00-common/02.bronze-helpers"

In [0]:

source_file = f"{landing_folder_path}/{v_batch_id}/drivers.json"

target_path = "s3://bizf1project123/Bronze/Aaisha/incremental-load/drivers"

In [0]:
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType
)

drivers_schema = StructType([
    StructField("dateOfBirth", StringType(), True),
    StructField("driverId", StringType(), True),
    StructField("name", StructType([
        StructField("familyName", StringType(), True),
        StructField("givenName", StringType(), True)
    ]), True),
    StructField("nationality", StringType(), True),
    StructField("url", StringType(), True)
])

In [0]:
drivers_df = (
    spark.read
        .format("json")
        .schema(drivers_schema)
        .option("mode", "FAILFAST")
        .load(source_file)
)

In [0]:
display(drivers_df)

In [0]:
drivers_final_df = add_ingestion_metadata(drivers_df)

In [0]:
display(drivers_final_df)

In [0]:
write_to_bronze(
    input_df=drivers_final_df,
    target_path=target_path,
    batch_id=v_batch_id
)